Generación CSV Maestro ADNI

Generamos un csv combinando los metadatos de las imágenes ADNI del repositorio con el estudio de las lesiones (filtrando aquellos que tienen CMBs definite).

El csv maestro con toda la información es:
/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_Master_Unified_Dataset.csv

De ahí, también se extrae por muestreo estratificado 250 imágenes con CMBs positivos:
/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/selected_positives_ADNI.csv

In [16]:
import pandas as pd
import os

# 1. Configuración de nombres de archivo y columnas
archivo_a = '/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_original_dataset_downloaded/MAYOADIRL_MRI_MCH_12Feb2026.csv'
archivo_b = '/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_original_dataset_downloaded/idaSearch_2_19_2026.csv'

columna_id_a = 'LONI_IMG_ID' 
columna_id_b = 'Image ID'

# 2. Carga de datos con detección de separador
try:
    df1 = pd.read_csv(archivo_a, sep=None, engine='python')
    df2 = pd.read_csv(archivo_b, sep=None, engine='python')
    
    # Limpiamos posibles espacios en los nombres de las columnas (para evitar el KeyError)
    df1.columns = df1.columns.str.strip()
    df2.columns = df2.columns.str.strip()
    
    print("Archivos cargados. Columnas en B detectadas:", df2.columns.tolist())
    
except Exception as e:
    print(f"Error al cargar archivos: {e}")
    raise

# 3. NORMALIZACIÓN DE IDs (Agregado por mí)
# En el archivo A: Convertimos a string, quitamos la 'I' del principio y limpiamos espacios
df1[columna_id_a] = df1[columna_id_a].astype(str).str.replace(r'^I', '', regex=True).str.strip()

# En el archivo B: Aseguramos que sea string y limpiamos espacios
df2[columna_id_b] = df2[columna_id_b].astype(str).str.strip()

# 4. Realizar el Join (Cruce)
df_inner = pd.merge(df1, df2, left_on=columna_id_a, right_on=columna_id_b, how='inner')

# 5. Definir filtros de limpieza
# Filtro A: TYPE MCH y STATUS Definite
condicion_mch = (df_inner['TYPE'] == 'MCH') & (df_inner['STATUS'] == 'Definite')

# Filtro B: TYPE en blanco (nulo o vacío)
# .isna() detecta NaNs, y el chequeo de string vacío detecta celdas sin texto
condicion_sanos = (df_inner['TYPE'].isna()) | (df_inner['TYPE'].astype(str).str.strip() == '') | (df_inner['TYPE'].astype(str) == 'nan')

# Aplicar el filtrado final: Solo nos quedamos con una de estas dos opciones
df_filtrado = df_inner[condicion_mch | condicion_sanos].copy()

# 6. Conteos de IDs ÚNICOS (Individuos/Sesiones)
ids_mch_unicos = df_filtrado[condicion_mch][columna_id_a].nunique()
ids_sanos_unicos = df_filtrado[condicion_sanos][columna_id_a].nunique()

# 7. Conteo de repeticiones (Número de MCH por cada ID)
# Esto nos dice cuántos MCH tiene cada ID de los que son 'Definite'
conteo_repeticiones = df_filtrado[condicion_mch].groupby(columna_id_a).size()

# --- RESULTADOS ---
print(f"--- RESULTADOS DEL FILTRADO ---")
print(f"IDs con MCH 'Definite' (pacientes con hallazgos): {ids_mch_unicos}")
print(f"IDs con TYPE en blanco (presuntos sanos): {ids_sanos_unicos}")
print(f"Total de filas tras la limpieza: {len(df_filtrado)}")

print(f"\n--- DISTRIBUCIÓN DE MCH POR ID")
if not conteo_repeticiones.empty:
    print(conteo_repeticiones.sort_values(ascending=False))
    print(f"\nTotal de filas de MCH (incluyendo repeticiones): {conteo_repeticiones.sum()}")
else:
    print("No se encontraron registros que cumplan la condición de MCH Definite.")

# 8. Análisis de Slice Thickness sobre los filtrados (agregado por mí)
if 'Imaging Protocol' in df_filtrado.columns:
    df_filtrado['Slice_Value'] = df_filtrado['Imaging Protocol'].str.extract(r'Slice Thickness=([0-9.]+)')
    print("\n--- SLICE THICKNESS EN DATASET LIMPIO (Valores más comunes) ---")
    print(df_filtrado['Slice_Value'].value_counts().head(5))


# 9. Guardar resultados
ruta_base = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/"
nombre_archivo = "ADNI_MCH_Clean_Dataset.csv"
ruta_completa = os.path.join(ruta_base, nombre_archivo)

# 2. Guardar el DataFrame filtrado
# Usamos index=False para no añadir una columna de números extra
df_filtrado.to_csv(ruta_completa, index=False)

print(f"Dataset guardado con éxito en: {ruta_completa}")

# En lugar de df_filtrado[condicion_mch], lo correcto para evitar el warning es:
# ids_mch_unicos = df_filtrado.loc[df_filtrado['TYPE'] == 'MCH', columna_id_a].nunique()

Archivos cargados. Columnas en B detectadas: ['Subject ID', 'Phase', 'Sex', 'Weight', 'Research Group', 'Visit', 'Archive Date', 'Study Date', 'Age', 'Modality', 'Description', 'Imaging Protocol', 'Image ID']
--- RESULTADOS DEL FILTRADO ---
IDs con MCH 'Definite' (pacientes con hallazgos): 1812
IDs con TYPE en blanco (presuntos sanos): 3868
Total de filas tras la limpieza: 15242

--- DISTRIBUCIÓN DE MCH POR ID
LONI_IMG_ID
460695     539
399542     525
378882     483
358741     461
342864     458
          ... 
982445       1
982371       1
981037       1
1017752      1
1017739      1
Length: 1812, dtype: int64

Total de filas de MCH (incluyendo repeticiones): 11374

--- SLICE THICKNESS EN DATASET LIMPIO (Valores más comunes) ---
Slice_Value
4.0    15076
5.0      164
4.5        2
Name: count, dtype: int64


/tmp/ipykernel_153267/1665609147.py:48: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ids_mch_unicos = df_filtrado[condicion_mch][columna_id_a].nunique()
/tmp/ipykernel_153267/1665609147.py:49: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ids_sanos_unicos = df_filtrado[condicion_sanos][columna_id_a].nunique()
/tmp/ipykernel_153267/1665609147.py:53: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  conteo_repeticiones = df_filtrado[condicion_mch].groupby(columna_id_a).size()


Dataset guardado con éxito en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_MCH_Clean_Dataset.csv


## JOIN csv

Hacemos join del csv de MAYO CLINIC, de MCH clean dataset y de study definite de marzo:

In [4]:
import pandas as pd
import numpy as np
import os

# --- RUTAS ---
path_mayo = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_original_dataset_downloaded/MAYOADIRL_MRI_MCH_12Feb2026.csv"
path_study = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_study_definiteCMB_3_26_2026.csv"
path_clean = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_MCH_Clean_Dataset.csv"
path_output = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_Master_Unified_Dataset.csv"

def clean_id(series):
    return series.astype(str).str.replace('I', '').str.split('.').str[0].str.strip()

print("Cargando archivos...")

# 1. Base Mayo (Mantenemos todas las filas: una por cada coordenada/lesión)
df_base = pd.read_csv(path_mayo)
df_base['MATCH_ID'] = clean_id(df_base['LONI_IMG_ID'])

# 2. Estudio (Demográficos)
# Deduplicamos por MATCH_ID manteniendo todas las columnas de la primera aparición
df_study = pd.read_csv(path_study)
df_study['MATCH_ID'] = clean_id(df_study['Image Data ID'])
df_study_unique = df_study.drop_duplicates(subset=['MATCH_ID']).drop(columns=['Image Data ID'], errors='ignore')

# 3. Clean Dataset (Metadatos técnicos y adicionales)
# Deduplicamos por MATCH_ID manteniendo TODAS las columnas técnicas del scan
df_clean = pd.read_csv(path_clean)
df_clean['MATCH_ID'] = clean_id(df_clean['LONI_IMG_ID'])
df_clean_unique = df_clean.drop_duplicates(subset=['MATCH_ID'])

# Identificamos columnas que ya existen en df_base para evitar duplicidad de nombres en el join
# pero permitimos que el merge traiga el resto de columnas nuevas
cols_overlap = [c for c in df_clean_unique.columns if c in df_base.columns and c != 'MATCH_ID']
df_clean_unique = df_clean_unique.drop(columns=cols_overlap)

print("Realizando la unión (JOIN) total...")

# Unimos Mayo con Estudio
df_unified = pd.merge(df_base, df_study_unique, on='MATCH_ID', how='left')

# Unimos con TODA la información del Clean Dataset (Metadatos técnicos, protocolos, etc.)
df_unified = pd.merge(df_unified, df_clean_unique, on='MATCH_ID', how='left')

print("Aplicando filtros de calidad...")

# Lógica de filtrado solicitada:
# - Sanos (TYPE vacío/NaN) 
# - O bien (TYPE MCH Y STATUS Definite)
cond_sanos = df_unified['TYPE'].isna()
cond_mch_definite = (df_unified['TYPE'] == 'MCH') & (df_unified['STATUS'] == 'Definite')

df_final = df_unified[cond_sanos | cond_mch_definite].copy()

# Eliminamos la columna auxiliar de cruce
df_final = df_final.drop(columns=['MATCH_ID'])

print(f"Guardando CSV unificado en: {path_output}")
df_final.to_csv(path_output, index=False)

print(f"--- RESULTADOS DEL PROCESO ---")
print(f"Filas resultantes: {len(df_final)}")
print(f"Columnas totales integradas: {len(df_final.columns)}")

Cargando archivos...
Realizando la unión (JOIN) total...
Aplicando filtros de calidad...
Guardando CSV unificado en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_Master_Unified_Dataset.csv
--- RESULTADOS DEL PROCESO ---
Filas resultantes: 15260
Columnas totales integradas: 56


In [ ]:
import pandas as pd
import numpy as np
import os

# Configuración
RANDOM_SEED = 42
TARGET_SAMPLE_SIZE = 250
CSV_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI/ADNI_Master_Unified_Dataset.csv"

# 1. Cargar datos
df_metadata = pd.read_csv(CSV_PATH)

# Función para extraer metadatos técnicos de las columnas 'Unnamed'
def extract_tech_info(row):
    info = {'MagneticField': None, 'Manufacturer': None}
    # Buscamos en todas las columnas de la fila por si la posición varía
    for val in row.values:
        val_str = str(val)
        if 'Field Strength=' in val_str:
            info['MagneticField'] = val_str.split('=')[1]
        elif 'Manufacturer=' in val_str:
            info['Manufacturer'] = val_str.split('=')[1]
    return pd.Series(info)

# 2. Agrupar por scan (Image ID) para tener una lista de sujetos únicos
# Tomamos la primera entrada de cada scan para los metadatos
df_scans = df_metadata.groupby('LONI_IMG_ID_STR').first().reset_index()

# 3. Limpieza y extracción de columnas necesarias
print("Extrayendo metadatos técnicos...")
tech_metadata = df_scans.apply(extract_tech_info, axis=1)
df_scans['MagneticField'] = tech_metadata['MagneticField']
df_scans['Manufacturer'] = tech_metadata['Manufacturer']
df_scans['Age'] = df_scans['Age_x']  # Usamos Age_x que es la columna poblada

# Como este CSV solo tiene positivos, definimos Has_CMB como True
df_scans['Has_CMB'] = True

# --- SELECCIÓN ESTRATIFICADA (Solo para Positivos en este CSV) ---

def perform_stratified_selection(df, n_target):
    # Binning de edad (60-70, 70-80, 80+)
    df = df.copy()
    df['Age_Bin'] = pd.cut(df['Age'], bins=[0, 70, 80, 120], labels=['<70', '70-80', '80+'])
    
    # Clave de estrato: Campo Magnético + Fabricante + Edad
    df['Stratum'] = (
        df['MagneticField'].astype(str) + "_" + 
        df['Manufacturer'].astype(str) + "_" + 
        df['Age_Bin'].astype(str)
    )
    
    # Muestreo estratificado
    # Nota: Si un estrato tiene pocos sujetos, pandas.sample puede fallar si no se gestiona.
    # Usamos groupby().apply() para un muestreo proporcional simplificado
    try:
        selection = df.groupby('Stratum', group_keys=False).apply(
            lambda x: x.sample(n=int(np.ceil(len(x) / len(df) * n_target)), random_state=RANDOM_SEED)
        ).head(n_target)
    except ValueError:
        # Fallback si algún estrato es demasiado pequeño
        selection = df.sample(n=n_target, random_state=RANDOM_SEED)
        
    return selection

# 4. Seleccionar los 250 positivos reales
df_pos_final = perform_stratified_selection(df_scans, TARGET_SAMPLE_SIZE)

print(f"Seleccionados {len(df_pos_final)} scans positivos reales para el Trial 1.")

# --- NOTA SOBRE LOS NEGATIVOS ---
# Dado que el CSV adjunto NO contiene imágenes sanas (negativas),
# para el Trial 2 hay que cargar la lista de sujetos sanos y aplicar el mismo
# procedimiento de 'extract_tech_info' y 'perform_stratified_selection'.


# Guardar lista de IDs seleccionados para procesar
df_pos_final[['LONI_IMG_ID_STR', 'MagneticField', 'Manufacturer', 'Age']].to_csv("selected_positives_ADNI.csv", index=False)

Extrayendo metadatos técnicos...
Seleccionados 250 scans positivos reales para el Trial 1.


/tmp/ipykernel_431924/3158678169.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selection = df.groupby('Stratum', group_keys=False).apply(
